### Задание 1. Парсинг таблицы (0.5 балла)

Извлеките таблицу с курсом валют с сайта Банка России https://www.cbr.ru/currency_base/daily/

Можно собирать теги таблицы через requests и BeautifulSoup циклом (сложнее) **ИЛИ** использовать pandas (удобнее, в одну строку)

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import json
import datetime
from time import sleep

In [51]:
table = pd.read_html('https://www.cbr.ru/currency_base/daily/')

df = table[0]
df.head()

,Цифр. код,Букв. код,Единиц,Валюта,Курс
0,36,AUD,1,Австралийский доллар,514324
1,944,AZN,1,Азербайджанский манат,420901
2,12,DZD,100,Алжирских динаров,537635
3,51,AMD,100,Армянских драмов,194253
4,764,THB,10,Батов,219589


### Задание 2. Парсинг requests + BeautifulSoup (3 балла)

С главной страницы сайта: https://naked-science.ru/ извлеките новости.

1. Передайте ссылку в requests и BeautifulSoup
2. Пройдите циклом по всем тегам 'div' с атрибутами class="news-item grid" и извлеките из каждого:
    - заголовок ('h3')
    - текст превью ('p')
    - автора ('div', class="meta-item_author")
    - ссылку на новость ('a', class="animate-custom") - не забудьте, что сама ссылка хранится в атрибуте 'href', нужно ее извлечь: .get('href')
    - дату (определите тег и атрибут самостоятельно)
    - просмотры (определите тег и атрибут самостоятельно)
    - теги (определите тег и атрибут самостоятельно), их можно собрать из общего тега и предобработать: заменить '#' на пустые кавычки и разделить .split() либо использовать вложенный цикл.
3. Соберите результат в датафрейм

Пример работы вашего цикла:

```
for tag in soup.find_all('div', {'class':"news-item grid"}):
    title = tag.find('h3').text
```

In [52]:
# your code here
url = 'https://naked-science.ru/'
response = requests.get(url)
response.encoding = 'utf-8'

soup = BeautifulSoup(response.text, 'html.parser')

result = []
for tag in soup.find_all('div', {'class': 'news-item grid'}):
    tags = tag.find_all('div', {'class': 'terms-item'})
    tags_res = [i.find('a').text.replace('#', '').replace('\n', '').strip() for i in tags]
    result.append({
        'title': tag.find('h3').find('a').text,
        'preview': tag.find('p').text,
        'author': tag.find('div', {'class': 'meta-item_author'}).text.replace('\n', '').strip(),
        'link': tag.find('a', {'class': 'animate-custom'}).get('href'),
        'date': tag.find('span', {'class': 'echo_date'}).text,
        'views': tag.find('span', {'class': 'fvc-count'}).text,
        'tags': tags_res
        # 'tags': tag.find('div', {'class': 'terms-item'}).find('a').text
    })

df = pd.DataFrame(result)
df.head()

,title,preview,author,link,date,views,tags
0,Археологи нашли первое свидетельство насилия н...,"Исследователи изучилили скелет лангобардки, ко...",Игорь Байдов,https://naked-science.ru/article/archeology/pe...,"2 июня, 11:01",472,"[Археология, женщины, насилие, Средневековая Е..."
1,Древнеегипетский флакон с подводкой для глаз з...,Во время раскопок в римском лагере в Йорке (Ан...,Татьяна Зайцева,https://naked-science.ru/article/archeology/dr...,"2 июня, 10:44",635,"[Археология, археологические раскопки, Древний..."
2,"Ученые выяснили, какие звуки русского языка лю...",Возьмите два вымышленных слова: «буола» и «кик...,ПНИПУ,https://naked-science.ru/article/column/zvuki-...,"2 июня, 10:36",155,"[ПНИПУ, восприятие звуков, русский язык, филол..."
3,Сверхмассивные черные дыры назвали крупнейшим ...,В популярной литературе сверхмассивные черные ...,Александр Березин,https://naked-science.ru/article/astronomy/sve...,"2 июня, 09:52",808,"[Астрономия, астрономия, астрофизика, сверхмас..."
4,Неандертальская ДНК сделала человека уязвимее ...,"Гены, унаследованные нами от неандертальцев и ...",Максим Абдулаев,https://naked-science.ru/article/biology/neand...,"2 июня, 09:32",223,"[Биология, антропогенез, иммунитет, неандертал..."


### Задание 3. Вакансии (1.5 балла)

1. Обратитесь к API портала «Работа России» (https://trudvsem.ru/opendata/api) для поиска вакансий.
2. Используйте requests.get(), преобразуйте ответ в json (используйте параметры при запросе: в "text" передайте "Python", "limit" - извлечем 100 записей).
3. Извлеките данные из ответа API. Сохраните собранную информацию в датафрейм.

In [53]:
url = 'http://opendata.trudvsem.ru/api/v1/vacancies'
params = {
    'text': 'Python',
    'limit': 100
}
result = []

response = requests.get(url, params=params)
data = response.json()
vacancies = data['results']['vacancies']
for i in vacancies:
    vacancy = i['vacancy']
    result.append({
        'job': vacancy['job-name'],
        'region': vacancy['region']['name'],
        'company': vacancy['company']['name'],
        'salary': f'{vacancy.get('salary_min')}-{vacancy.get('salary_max')}',
        'employment': vacancy.get('employment'),
        'schedule': vacancy.get('schedule'),
        'qualification': vacancy.get('qualification')


    })
df = pd.DataFrame(result)
df.head()

,job,region,company,salary,employment,schedule,qualification
0,Программист Python,Ярославская область,"ООО ""ИНФОРМАЦИОННЫЕ ТЕХНОЛОГИИ""",80000-180000,Удалённая,Полный рабочий день,-
1,Разработчик Python,Город Москва,"ООО ""РДП.РУ""",80000-120000,NaN,Полный рабочий день,Средняя
2,Программист Python,Город Москва,"ООО НПО ""ТУРБУЛЕНТНОСТЬ-ДОН""",45000-45000,NaN,Полный рабочий день,Высокая
3,Программист Python,Город Москва,"АО ""ГРИНАТОМ""",180000-250000,Удалённая,Ненормированный рабочий день,Мидл+
4,Программист Python,Город Москва,"ООО ""НТЦ ""ВУЛКАН""",180000-200000,NaN,Полный рабочий день,Программист


### Задание 4. Землетрясения в мае 2026 (3 балла)

Работаем с API USGS Earthquake Catalog, [документация](https://earthquake.usgs.gov/fdsnws/event/1/)

URL для получения данных: https://earthquake.usgs.gov/fdsnws/event/1/query

Параметры запроса (обязательные):

- format - "geojson"
- starttime – начальная дата (YYYY-MM-DD, возьмем 1 мая 2026)
- endtime – конечная дата (YYYY-MM-DD, возьмем 31 мая 2026)
- minmagnitude – минимальная магнитуда (возьмем 4.5)

Поля (ключи), которые необходимо извлечь из каждого землетрясения:

- mag - Магнитуда
- place - Описание места
- time  - Дата и время события, позже нужно   перевести из миллисекунд в читаемый формат, пример:
```
import datetime

time_ms = 1748419123460
date_obj = datetime.datetime.fromtimestamp(time_ms / 1000)
date_str = date_obj.strftime('%Y-%m-%d')
print(date_str)  # вернет 2025-05-28
```
- felt  - количество людей, почувствовавших землетрясение
- mmi   - интенсивность (измеренная инструментально)
- sig   - интенсивность по показаниям людей
- alert - уровень оповещения (green, yellow, red или None)

**Все перечисленные выше поля извлекаются по ключу `'properties'`**

- longitude, latitude, depth - координаты и глубина (км)

**Извлекаются по ключам '`geometry'` -> `'coordinates'`**, получите список [longitude, latitude, depth]

Чтобы собрать датафрейм из json, нужно:
1. Пройти по **списку словарей, ключ `features`** (это и есть землетрясения) и для каждого извлечь все перечисленные поля.
2. Сложить данные в список словарей или список списков
3. Собрать датафрейм

In [54]:
import datetime

url = 'https://earthquake.usgs.gov/fdsnws/event/1/query'
params = {'format': 'geojson',
    'starttime': '2026-05-01',
    'endtime': '2026-05-31',
    'minmagnitude': 4.5
}

response = requests.get(url, params=params)
data = response.json()
print(data)
result = []
features = data['features']
for feature in features:
    properties = feature['properties']
    geometry = feature['geometry']['coordinates']
    date_obj = datetime.datetime.fromtimestamp(properties['time'] / 1000)
    date_str = date_obj.strftime('%Y-%m-%d')
    result.append({
        'mag': properties['mag'],
        'place': properties['place'],
        'time': date_str,
        'felt': properties['felt'],
        'mmi': properties['mmi'],
        'sig': properties['sig'],
        'alert': properties['alert'],
        'longtitude': geometry[0],
        'latitude': geometry[1],
        'depth': geometry[2]
    })
df = pd.DataFrame(result)
df.head()

{'type': 'FeatureCollection', 'metadata': {'generated': 1780398395000, 'url': 'https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2026-05-01&endtime=2026-05-31&minmagnitude=4.5', 'title': 'USGS Earthquakes', 'status': 200, 'api': '2.4.0', 'count': 426}, 'features': [{'type': 'Feature', 'properties': {'mag': 5, 'place': '81 km SE of Lae, Papua New Guinea', 'time': 1780181151647, 'updated': 1780183425040, 'tz': None, 'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/us7000spkp', 'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=us7000spkp&format=geojson', 'felt': None, 'cdi': None, 'mmi': None, 'alert': None, 'status': 'reviewed', 'tsunami': 0, 'sig': 385, 'net': 'us', 'code': '7000spkp', 'ids': ',us7000spkp,', 'sources': ',us,', 'types': ',origin,phase-data,', 'nst': 61, 'dmin': 2.115, 'rms': 0.77, 'gap': 62, 'magType': 'mb', 'type': 'earthquake', 'title': 'M 5.0 - 81 km SE of Lae, Papua New Guinea'}, 'geometry': {'type': 'Point', 'coordi

,mag,place,time,felt,mmi,sig,alert,longtitude,latitude,depth
0,5.0,"81 km SE of Lae, Papua New Guinea",2026-05-31,NaN,NaN,385,NaN,147.4544,-7.2968,68.854
1,5.6,"137 km SSE of Oistins, Barbados",2026-05-31,22.0,3.194,492,green,-59.1029,11.9084,35.000
2,4.9,"24 km NNW of Murghob, Tajikistan",2026-05-31,NaN,NaN,369,NaN,73.8219,38.3564,131.794
3,4.5,"26 km SE of Ashoro, Japan",2026-05-30,NaN,NaN,312,NaN,143.7857,43.0767,116.722
4,4.6,"63 km W of Catuday, Philippines",2026-05-30,NaN,NaN,326,NaN,119.2212,16.3879,10.000


### Задание 5. Всемирный банк (дополнительно, 2 балла)
Если нет доступа к API Всемирного банка, то:
1) Выберите любой датасет / API
2) Выделите примерную структуру (какие признаки, объем, есть ли ограничения с доступом и т.д.)
3) Ответьте на вопрос: Можно ли обучить модель (регрессия / классификация)?

API Всемирного банка (World Bank Open Data), URL для получения данных: https://api.worldbank.org/v2

Загрузите данные по выбранной стране (можно использовать код страны, например "RU" для России, "US" для США, "CN" для Китая и т.д.) за период с 2000 по 2024 год.

Соберите следующие показатели (коды индикаторов Всемирного банка):

- NY.GDP.MKTP.CD – ВВП (в долларах США)
- NY.GDP.PCAP.CD – ВВП на душу населения
- NY.GDP.MKTP.KD.ZG – Годовой рост ВВП (%)
- FP.CPI.TOTL.ZG – Инфляция (дефлятор ВВП, %)
- SL.UEM.TOTL.NE.ZS – Уровень безработицы (% от общей численности рабочей силы)
- SE.XPD.TOTL.GD.ZS – Расходы на образование (% от ВВП)
- SE.TER.ENRR – Валовой коэффициент охвата высшим образованием (%)
- SP.POP.TOTL – Общая численность населения
- SP.DYN.LE00.IN – Ожидаемая продолжительность жизни при рождении (лет)

Чтобы собрать датафрейм, нужно:
- для каждого индикатора выполнить GET-запрос к API, параметры: "format" - "json", "date" - "2000:2024"
- из ответа API извлечь значения индикатора (ключ "value") за 2000-2024 гг (ключ "date").
- объединить в один датафрейм: строки — годы, столбцы — названия индикаторов (переименуйте для удобства).

**Формат ссылки для запроса**: базовая_ссылка/код_страны/indicator/код_индикатора

Важно:
- API Всемирного банка иногда возвращает пустые значения. Для некоторых показателей данные могут быть не за все годы.
- Запросы лучше выполнять с паузой (не обязательно, но вежливо).

In [ ]:
indicators = ['NY.GDP.MKTP.CD', 'NY.GDP.PCAP.CD', 'NY.GDP.MKTP.KD.ZG', 'FP.CPI.TOTL.ZG', 'SL.UEM.TOTL.NE.ZS', 'SE.XPD.TOTL.GD.ZS', 'SE.TER.ENRR', 'SP.POP.TOTL', 'SP.DYN.LE00.IN']

# your code here
